# Sequential SBI benchmark on Kaggle

This notebook is self-contained: it installs the project and defines all task and benchmark configurations below. It does not require uploading or attaching a Kaggle Dataset. In Kaggle notebook settings, select a GPU accelerator and enable Internet before running all cells.

**Run this file as a Kaggle Notebook using Kaggle's notebook UI. Do not pass the `.ipynb` to `kagglex run --file`: that option is for Python scripts and tries to execute the notebook's JSON as Python.**


The default smoke run uses 40 inference simulations per target, one target replicate, and 64 posterior draws. Keep budgets at or below 100 for a short CPU/GPU smoke test. Increase `BUDGETS` and `REPLICATES` only for longer experiments.

The install cell deliberately does not reinstall `torch` or `torchvision`; Kaggle supplies a CUDA-enabled PyTorch build.

In [ ]:
%pip install -q "git+https://github.com/nidropakshin2/Sequential-FMPE.git#subdirectory=code" "sbi==0.21.0" pandas matplotlib torchdiffeq PyYAML

In [ ]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: this run will use CPU. Select a GPU accelerator in Kaggle settings for faster training.")

import sfmpe
print("SFMPE package:", sfmpe.__file__)

## Task configurations

Each task config is an ordinary Python dictionary, in the same form as the `config` dictionaries in `tutorial.ipynb` and `stochvol.ipynb`. Vector and matrix parameters are written as Python lists. The benchmark writes these dictionaries to temporary YAML files because the shared runner consumes YAML task configs.

In [ ]:
TASK_CONFIGS = {
    "Gaussian": {
        "prior": {
            "mean": [0.0, 0.0],
            "covariance": [[1.5, 0.3], [0.3, 1.0]],
        },
        "simulator": {
            "likelihood_shift": [0.0, 0.0],
            "likelihood_covariance": [[0.4, 0.1], [0.1, 0.6]],
        },
        "summary": {"type": "identity"},
        "logger": {"name": "Gaussian", "level": "WARNING", "log_to_file": False},
    },
    "SIR": {
        "prior": {
            "gamma_range": [0.05, 0.5],
            "beta_range": [0.5, 5.0],
        },
        "simulator": {
            "S": 1000,
            "I": 100,
            "R": 0,
            "T": 40,
        },
        "summary": {
            "type": "handmade",
            "input_dim": 3,
            "hidden_dim": 32,
            "num_layers": 2,
            "output_dim": 5,
            "dropout": 0.1,
        },
        "logger": {"name": "SIR", "level": "WARNING", "log_file_path": "/kaggle/working/sir.log"},
    },
    "StochVol": {
        "prior": {
            "mu_loc": 0.0,
            "mu_scale": 0.2,
            "phi_alpha": 20.0,
            "phi_beta": 1.5,
            "sigma_ig_shape": 2.5,
            "sigma_ig_scale": 0.025,
            "m_loc": -1.5,
            "m_scale": 1.0,
        },
        "simulator": {"T": 64},
        "summary": {"type": "handmade"},
        "logger": {"name": "StochVol", "level": "WARNING", "log_to_file": False},
    },
}

SFMPE_CONFIG = {
    "flow": {
        "init_dist": {"type": "normal"},
        "velocity_model": {"type": "simple", "params": {"hidden_dim": 32, "num_layers": 2}},
        "path": {"type": "affine"},
    },
    "optimizer": {
        "type": "Adam",
        "params": {
            "task": {"summary": {"lr": 0.0001}},
            "flow_model": {"lr": 0.0001},
        },
    },
    "loss": "MSELoss",
    "proposal": {"method": "Truncated", "n_steps": 8, "params": {"quantile": 0.5}},
    "logger": {"name": "Benchmark", "level": "WARNING", "log_to_console": False, "log_to_file": False},
    "run_sequential": {
        "num_rounds": 2,
        "sims_per_round": 20,
        "sims_per_sample": 1,
        "clean_sampling": False,
        "upd_x": False,
        "training": {
            "epochs": 8,
            "show_every": 1000,
            "batch_size": 32,
            "shuffle": True,
            "num_workers": 0,
        },
    },
}

## Benchmark matrix and run settings

Edit the lists below to select tasks, inference methods, simulation budgets, and repetitions. `SMC_ABC` requires a budget of at least 20. `SFMPE` reserves a small part of each budget for setup.

In [ ]:
TASKS = ["Gaussian", "SIR", "StochVol"]
METHODS = ["SFMPE", "SNPE_C", "SMC_ABC"]
BUDGETS = [40]
REPLICATES = 1
ROUNDS = 2
POSTERIOR_SAMPLES = 64
SEED = 2026
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRAINING = {
    "epochs": 8,
    "stop_after_epochs": 3,
    "batch_size": 32,
    "learning_rate": 0.0005,
    "validation_fraction": 0.2,
}
SMC_ABC = {"particles": 20, "initial_population": 20, "epsilon_decay": 0.5}
DIAGNOSTICS = {
    "ppc_samples": 16,
    "sensitivity_enabled": False,
    "sensitivity_scale": 0.1,
    "sensitivity_samples": 16,
    "sbc_c2st": False,
}

if any(int(budget) > 100 for budget in BUDGETS):
    raise ValueError("For a Kaggle smoke run, keep every inference budget at or below 100.")
if "SMC_ABC" in METHODS and any(int(budget) < 20 for budget in BUDGETS):
    raise ValueError("SMC_ABC requires every budget to be at least 20.")
if not TASKS or not METHODS or not BUDGETS:
    raise ValueError("TASKS, METHODS, and BUDGETS must each contain at least one value.")
if set(TASKS) - set(TASK_CONFIGS):
    raise ValueError(f"Missing task configuration(s): {sorted(set(TASKS) - set(TASK_CONFIGS))}")

print(f"Device: {DEVICE}; tasks={TASKS}; methods={METHODS}; budgets={BUDGETS}; replicates={REPLICATES}")

## Run benchmark and build reports

The runner writes one CSV for per-target results and one for aggregates, plus a Markdown report and metric plots. Output is stored under `/kaggle/working/sfmpe-benchmark` and zipped in the final cell for download.

In [ ]:
from pathlib import Path
import yaml

from sfmpe.benchmark.report import export_benchmark_results
from sfmpe.benchmark.runner import run_benchmark

CONFIG_ROOT = Path("/kaggle/working/sfmpe-benchmark-configs")
OUTPUT_DIR = Path("/kaggle/working/sfmpe-benchmark")
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)

for task_name in TASKS:
    task_path = CONFIG_ROOT / f"{task_name.lower()}.yaml"
    task_path.write_text(yaml.safe_dump(TASK_CONFIGS[task_name], sort_keys=False), encoding="utf-8")

sfmpe_path = CONFIG_ROOT / "sfmpe.yaml"
sfmpe_path.write_text(yaml.safe_dump(SFMPE_CONFIG, sort_keys=False), encoding="utf-8")

BENCHMARK_CONFIG = {
    "seed": SEED,
    "device": DEVICE,
    "tasks": {name: f"{name.lower()}.yaml" for name in TASKS},
    "methods": METHODS,
    "budgets": BUDGETS,
    "replicates": REPLICATES,
    "rounds": ROUNDS,
    "posterior_samples": POSTERIOR_SAMPLES,
    "output": str(OUTPUT_DIR),
    "sfmpe_config": "sfmpe.yaml",
    "sfmpe_proposal_methods": {"StochVol": "NPE-A"},
    "training": TRAINING,
    "smc_abc": SMC_ABC,
    "diagnostics": DIAGNOSTICS,
}
benchmark_path = CONFIG_ROOT / "benchmark.yaml"
benchmark_path.write_text(yaml.safe_dump(BENCHMARK_CONFIG, sort_keys=False), encoding="utf-8")

print("Starting benchmark; this can take several minutes even with smoke-test budgets.")
summaries = run_benchmark(benchmark_path, output_dir=OUTPUT_DIR)
export_benchmark_results(OUTPUT_DIR)
print(f"Finished {len(summaries)} task/method/budget combinations.")
print("Results directory:", OUTPUT_DIR)

In [ ]:
from IPython.display import display, Markdown
import shutil

report_path = OUTPUT_DIR / "benchmark_report.md"
display(Markdown(report_path.read_text(encoding="utf-8")))

archive_path = Path("/kaggle/working/sfmpe-benchmark-report.zip")
shutil.make_archive(str(archive_path.with_suffix("")), "zip", OUTPUT_DIR)
print("Download the report archive from the notebook output:", archive_path)
print("Included files:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR))